In [ ]:
import requests 
import pandas as pd 
 
owner = "pandas-dev" 
repo = "pandas" 
token = "YOUR_GITHUB_TOKEN" 
 
url = f"https://api.github.com/repos/{owner}/{repo}/issues" 
 
headers = { 
   "Accept": "application/vnd.github+json" 
} 
 
if token != "YOUR_GITHUB_TOKEN": 
   headers["Authorization"] = f"Bearer {token}" 
 
params = { 
   "state": "all", 
   "per_page": 100, # max allowed 
   "page": 1 
} 
 
all_issues = [] 
 
while True: 
   response = requests.get(url, headers=headers, params=params, timeout=10) 
 
   if response.status_code == 403: 
       print("Rate limit exceeded:", response.json()) # 60 requests/hr if unauthenticated 
       break 
   elif response.status_code != 200:  
       print("Error:", response.status_code, response.text) 
       break 
 
   issues = response.json() 
 
   if not issues: 
       break 
 
   for issue in issues: 
       if "pull_request" in issue: 
           continue 
 
       all_issues.append({ 
           "id": issue.get("id"), 
           "issue_number": issue.get("number"), 
           "title": issue.get("title"), 
           "body": issue.get("body"), 
           "state": issue.get("state"), 
           "created_at": issue.get("created_at"), 
           "updated_at": issue.get("updated_at"), 
           "closed_at": issue.get("closed_at"), 
           "user_login": issue.get("user", {}).get("login"), 
           "comments_count": issue.get("comments"), 
           "labels": ", ".join([label["name"] for label in issue.get("labels", [])]), 
           "url": issue.get("html_url") 
       }) 
 
   params["page"] += 1 
 
df = pd.DataFrame(all_issues) 
df.to_csv("github_issues.csv", index=False, encoding="utf-8") 